# Machine Learning

Pada tahap ini dilakukan pembangunan model machine learning untuk memprediksi durasi perjalanan taksi (`duration_min`). Model akan dievaluasi menggunakan MAE, RMSE, dan R² Score.

Beberapa algoritma yang digunakan:

- Linear Regression
- Random Forest Regressor
- XGBoost Regressor


In [1]:
import sys

print(sys.executable)
print(sys.version)

/Users/retnolintang/Downloads/Taxi Trip Duration Predictions/.venv/bin/python
3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 21.0.0 (clang-2100.0.123.102)]


In [2]:
import xgboost

print(xgboost.__version__)

2.1.4


In [3]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [4]:
#Load data preprocessing
X_train = pd.read_csv('../Data/X_train.csv')
X_test = pd.read_csv('../Data/X_test.csv')
y_train = pd.read_csv('../Data/y_train.csv')
y_test = pd.read_csv('../Data/y_test.csv')

print(X_train.shape)
print(X_test.shape)

(3157358, 9)
(789340, 9)


In [5]:
#Ubah target menjadi 1 dimensi
y_train = y_train.squeeze()
y_test = y_test.squeeze()

print(y_train.shape)

(3157358,)


In [6]:
#Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

In [7]:
#Random Forest
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

In [8]:
#XGBoost
xgb = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)

In [9]:
#Evaluasi model
results = pd.DataFrame({
    'Model':[],
    'MAE':[],
    'RMSE':[],
    'R2':[]
})

models = {
    'Linear Regression': lr_pred,
    'Random Forest': rf_pred,
    'XGBoost': xgb_pred
}

for name, pred in models.items():

    mae = mean_absolute_error(y_test, pred)

    rmse = np.sqrt(mean_squared_error(y_test, pred))

    r2 = r2_score(y_test, pred)

    temp = pd.DataFrame({
        'Model':[name],
        'MAE':[mae],
        'RMSE':[rmse],
        'R2':[r2]
    })

    results = pd.concat(
        [results, temp],
        ignore_index=True
    )

results

,Model,MAE,RMSE,R2
0,Linear Regression,5.220091,7.623127,0.676619
1,Random Forest,3.596781,5.743809,0.816410
2,XGBoost,3.696599,5.835153,0.810525


In [10]:
#Urutkan model terbaik
results.sort_values(
    by='R2',
    ascending=False
)

,Model,MAE,RMSE,R2
1,Random Forest,3.596781,5.743809,0.816410
2,XGBoost,3.696599,5.835153,0.810525
0,Linear Regression,5.220091,7.623127,0.676619


## Insight

1. **XGBoost dipilih sebagai model final** karena memiliki performa yang sangat kompetitif dengan **R² = 0,8105, MAE = 3,70, dan RMSE = 5,84**, yang hanya sedikit berbeda dibandingkan Random Forest.

2. **Random Forest dan XGBoost sama-sama unggul dibandingkan Linear Regression**, menunjukkan bahwa model berbasis ensemble lebih mampu menangkap pola kompleks pada data durasi perjalanan taksi.

3. **Selisih performa antara Random Forest dan XGBoost relatif kecil**, sehingga pemilihan model tidak hanya didasarkan pada akurasi, tetapi juga mempertimbangkan **efisiensi komputasi** dan **kemudahan implementasi**.

4. **XGBoost lebih efisien untuk digunakan pada dataset berskala besar** karena memiliki optimasi komputasi yang lebih baik dan mampu memproses data dengan waktu yang lebih cepat dibandingkan Random Forest.

5. **Random Forest membutuhkan waktu komputasi yang sangat lama (lebih dari 2 jam)** pada proses training dan hyperparameter tuning, sehingga kurang efisien untuk pengembangan model secara berulang.

6. **XGBoost dipilih sebagai model final** karena memberikan keseimbangan yang baik antara **akurasi, efisiensi komputasi, dan skalabilitas**, sehingga lebih siap diterapkan pada skenario operasional nyata.